# Day 2 — Multi-turn Conversation

## My initial intuition

The Anthropic API is stateless — each request is independent and the
server holds no memory of past calls. But chat products like Claude.ai
behave as if they remember context. The reconciliation is simple but
non-obvious: **the client maintains the conversation history**.

To support N turns of dialogue, the client builds up a `messages` list
where every prior `user` and `assistant` message is included on every
new API call. Each request re-sends the entire history.

## What I learned today

- **Multi-turn = client-side state management.** No server-side memory;
  the client is fully responsible for retaining and re-sending history.
- **The user/assistant role alternation is an API protocol requirement,
  not a stylistic suggestion.** Anthropic API enforces strict alternation
  (e.g., `[user, assistant, user, assistant, ...]`). Two consecutive
  `user` messages will be rejected.
- **Why two roles exist at all**: Claude is trained to *only generate
  in the assistant position*. Past `assistant` messages are read as
  "things I already said" (don't re-answer); `user` messages are read
  as "what I need to respond to now". Without role separation, Claude
  would treat its prior answers as new prompts and produce nonsense.
- **Tokens grow O(n²) across rounds, not linearly.** Empirically:
  - Round 1: 16 input tokens
  - Round 2: 262 input tokens (≈ round 1 input + round 1 output + new question)
  - Round 5 estimate: ~1075 input tokens (cumulative ≈ 170× round 1)
  - This is the cost dynamic that motivates **prompt caching** (Phase 4)
    and **conversation summarization** (Phase 5).
- **`stop_reason` is the truth source for "did Claude finish?"** A
  function that returns just `.content[0].text` is hiding important
  signal. Production code should surface `stop_reason` to the caller.

## Design choices I made + reasoning

- **Three helper functions (`add_user_message`, `add_assistant_message`,
  `chat`)** — Even though the official Quickstart shows inline dicts,
  building a tiny abstraction layer makes the call site cleaner and
  prevents typos in role strings. DRY.
- **Globals for `model` and `max_tokens`** — Single source of truth.
  Want to swap models? One edit. Phase 4 (model tiering) will revisit
  whether globals are still the right pattern when different turns
  need different models.
- **Test with a follow-up question requiring context resolution**
  ("How is it used in most modern LLMs?" — where "it" only resolves
  to "transformer" if multi-turn works) — A test should be capable
  of failing. A useless test is one that passes when the system is
  broken.

## What confused me / still unsure about

- **The `chat()` function silently drops `stop_reason` and `usage`.**
  If `max_tokens` truncates the response, the caller has no idea.
  Production-quality designs: raise an exception, return a structured
  dict (`{text, truncated, usage, stop_reason}`), or use `logging`.
  `print()` is anti-pattern in library code — it writes to stdout
  unconditionally, regardless of context. NomNom's `client.py` uses
  `logging` — will examine that pattern in Day 6.
- **Exact mechanics of why 16 input tokens for "what is transformer..."**
  — Tokenization overhead + role metadata + maybe a system-default
  prefix. Need to read up on Anthropic's tokenizer specifics in Phase 3
  (it matters for embedding cost analysis).

## Self-check answers (the questions I had to answer)

1. **Why `user` and `assistant` roles?**
   API protocol enforcement (must alternate). Claude is trained to
   generate only in the `assistant` position; past `assistant` messages
   are treated as already-said, past `user` messages are treated as
   pending response targets. Without role distinction, Claude would
   either re-answer its own messages or treat its own output as new
   user input — both break the conversation model.

2. **Token cost in round 2? Round 4? Why prompt caching?**
   Measured: Round 1 input = 16 tokens, Round 2 input = 262 tokens.
   Each round includes all prior `user` + `assistant` messages, so
   cumulative input grows **O(n²)** with the number of turns, not
   linearly. By round 5, cumulative input ≈ 170× the first round.
   By round 50, you'd exceed model context window entirely. Prompt
   caching (Phase 4) lets the server reuse cached prefixes and only
   bill for the delta — typically 90% savings.

3. **What if `max_tokens=10`? How to fix `chat()`?**
   Response gets silently truncated, `stop_reason='max_tokens'`,
   caller has no idea. Fix options:
   - Raise an exception on truncation (caller decides)
   - Return a structured dict including `truncated` flag and metadata
   - Use `logging.warning` instead of `print` (production standard)
   - `print` is wrong in library code — it bypasses caller's logging
     strategy and pollutes stdout.

## Open questions for later phases

- **Phase 3 (RAG)**: Does the same O(n²) cost dynamic apply when
  injecting retrieved chunks into context? How does that change
  prompt caching strategy?
- **Phase 4 (caching)**: What's the actual cost ratio for a cached
  prefix vs. a regular one? Worth measuring on NomNom.
- **Phase 5 (workflow)**: Multi-turn dialogue is one kind of state;
  multi-step workflows are another. How are they similar / different?




## Interview Q&A

1. Why does message list need two types of role: "user" and "assistant"? What will happen if only using "user" role in the sequential conversation?

   To let the model know which messages are past assistant outputs (already said — don't re-answer) vs. user requests (need to respond to now). 
   
   Claude is trained to generate only in the assistant position, so the role distinction is what makes turn-taking work. 
   
   Also: Anthropic API enforces strict user/assistant alternation — two consecutive "user" messages will be rejected by the API. So this isn't just a stylistic suggestion; it's protocol-level.

2. What's the token cost in the 2nd round conversation? What about the 4th round? Why need prompt caching (dive deep at later phase 4)?

   Measured: Round 1 input = 16 tokens, Round 2 input = 262 tokens (≈16× because it includes round 1's full user msg + Claude's full answer + new question).
   
   Growth pattern is O(n²), not linear — each new round re-sends the entire history. By round 5, cumulative input ≈ 170× round 1. By round 50, you'd hit context window limits.
   
   Prompt caching (Phase 4) lets the server cache the unchanging prefix and only bill for the delta — typically 90% savings on long conversations.

3. What will happen if max_token=10? How to tailor chat function to avoid silent truncation issue?

   Response gets silently truncated; stop_reason becomes "max_tokens" but caller of chat() has no way to know — it just gets a half-finished string.
   
   Fix options (in order of professionalism):
   - Raise an exception on truncation → forces caller to handle it
   - Return a dict {text, truncated, stop_reason, usage} → caller chooses
   - Use logging.warning instead of print → production standard
   
   print() is anti-pattern in library code: it pollutes stdout and bypasses the caller's logging setup. NomNom's client.py uses logging — will revisit in Day 6.

In [1]:
# load env variable 
from dotenv import load_dotenv
load_dotenv()

# import Anthropic and create a Anthropic client
from anthropic import Anthropic
client = Anthropic()

In [2]:
# define 3 helper functions
def add_user_message(text, messages):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(text, messages):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)

def chat(model, max_tokens, messages):
    response = client.messages.create(
        model = model,
        max_tokens=max_tokens,
        messages=messages
    )
    print("Input tokens:", response.usage.input_tokens)
    print("Output tokens:", response.usage.output_tokens)
    return response.content[0].text

In [3]:
# put them all toghther
model="claude-haiku-4-5-20251001"
max_tokens = 1000
messages = []

add_user_message(text="what is transformer, give me concise answer", messages=messages)

answer = chat(model, max_tokens, messages)
print("answer: ", answer)

add_assistant_message(text=answer, messages=messages)

add_user_message(text="How is it used in most modern LLMs?", messages=messages)

answer2 = chat(model, max_tokens, messages)
print("answer 2: ", answer2)

Input tokens: 16
Output tokens: 231
answer:  # Transformer

A **transformer** is a deep learning architecture that processes data sequentially using **self-attention** mechanisms instead of recurrence.

## Key Features:
- **Self-attention**: Weighs relationships between all input elements simultaneously
- **Parallel processing**: Handles entire sequences at once (unlike RNNs)
- **Scalable**: Works well with large datasets and long sequences

## Main Components:
1. **Encoder**: Processes input
2. **Decoder**: Generates output
3. **Attention layers**: Learn which parts of input matter most
4. **Feed-forward networks**: Process attention outputs

## Why It's Important:
- Foundation for modern AI models (BERT, GPT, T5)
- Excellent for NLP, but also used in vision and multimodal tasks
- Enables transfer learning and pre-training

## Simple Analogy:
Imagine reading a sentence—you focus on different words depending on context. Self-attention does this automatically, learning which words to pa